# Online Retail ETL Pipeline

## Objectives

* Extract raw transaction data from the Online Retail.csv dataset
* Transform and clean the data (handle missing values, data type conversions, data validation)
* Perform exploratory analysis to understand data quality
* Load the cleaned data to the cleandata folder as a processed dataset

## Inputs

* Raw data source: Dataset/rawdata/Online Retail.csv

## Outputs

* Cleaned dataset: Dataset/cleandata/online_retail_cleaned.csv
* Data quality report with summary statistics and missing value analysis

## Additional Comments

* This notebook assumes the raw data file exists at Dataset/rawdata/Online Retail.csv
* The ETL is designed to handle common data quality issues in e-commerce transaction data

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [17]:
import os
current_dir = os.getcwd()
current_dir

'c:\\'

In [18]:
import os
current_dir = os.getcwd()                # -> 'c:\'
os.chdir(os.path.dirname(current_dir))   # parent of 'c:\' is still 'c:\'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [19]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [20]:
current_dir = os.getcwd()
current_dir

'c:\\'

# Extract: Load Raw Data

Load the raw data from the CSV file and perform initial inspection

In [21]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load raw data
raw_data_path = 'Dataset/rawdata/Online Retail.csv'
df = pd.read_csv(raw_data_path)

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nBasic info:")
print(df.info())

FileNotFoundError: [Errno 2] No such file or directory: 'Dataset/rawdata/Online Retail.csv'

## Conclusions and Next Steps

# Transform: Data Cleaning & Preprocessing

Handle missing values, convert data types, and validate data quality

In [ ]:
# Create a copy for transformation
df_clean = df.copy()

# Remove cancelled transactions (InvoiceNo starting with 'C')
print("Cancelled transactions before removal:", (df_clean['InvoiceNo'].astype(str).str.startswith('C')).sum())
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Check for missing values
print("Missing values before cleaning:")
print(df_clean.isnull().sum())
print(f"\nTotal missing values: {df_clean.isnull().sum().sum()}")

# Remove rows with missing values in critical columns
df_clean = df_clean.dropna(subset=['CustomerID', 'Description'])

# Handle missing values in other columns
df_clean['Country'].fillna('Unknown', inplace=True)

print(f"\nMissing values after cleaning:")
print(df_clean.isnull().sum())

# Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Convert numeric columns
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce')
df_clean['UnitPrice'] = pd.to_numeric(df_clean['UnitPrice'], errors='coerce')
df_clean['CustomerID'] = df_clean['CustomerID'].astype('int64')

# Remove rows with negative or zero quantities (likely returns or errors)
df_clean = df_clean[df_clean['Quantity'] > 0]

# Remove rows with negative or zero prices
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Calculate total sales amount
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f"\nDataset shape after transformation: {df_clean.shape}")
print(f"Rows removed: {len(df) - len(df_clean)}")
print(f"\nData types after transformation:")
print(df_clean.dtypes)

Cancelled transactions before removal: 9288
Missing values before cleaning:
InvoiceNo         0
StockCode         0
Description    1454
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Country           0
dtype: int64

Total missing values: 1454

Missing values after cleaning:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

Dataset shape after transformation: (530104, 9)
Rows removed: 11805

Data types after transformation:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
Country                object
TotalAmount           float64
dtype: object


---

## Data Quality Analysis

Analyze the cleaned data to verify quality and understand characteristics

# Load: Save Cleaned Data

Save the cleaned dataset to the cleandata folder

In [ ]:
# Create cleandata directory if it doesn't exist
import os
clean_data_dir = 'Dataset/cleandata'
os.makedirs(clean_data_dir, exist_ok=True)

# Save cleaned data
output_path = f'{clean_data_dir}/online_retail_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f"✓ Cleaned data saved to: {output_path}")
print(f"✓ File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")
print(f"✓ Number of rows: {len(df_clean):,}")
print(f"✓ Number of columns: {len(df_clean.columns)}")
print("\n=== ETL Pipeline Completed Successfully ===")

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'Dataset/cleandata'